# YOLO26n으로 원판 찾기

검증 영상 한 장에 표시한 정답과 사전학습 모델의 예측을 먼저 비교합니다. 이후 같은 장면에서 원판 학습 후의 변화를 살펴봅니다.

In [ ]:
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, DATASET, DEVICE, RUNS, SPLIT, VIDEOS, WEIGHTS
from wlcv.data.dataset import label_frame, prepare_dataset
from wlcv.data.frames import read_rgb
from wlcv.evaluation.plots import plot_comparison, plot_training, predict_rgb
from wlcv.training.train import train_baseline

## 1. 정답과 사전학습 예측 비교

초록색 테두리가 사람이 표시한 원판입니다. 오른쪽에는 COCO로 사전학습한 모델이 인식한 물체와 신뢰도가 표시됩니다. 다른 장면을 보려면 라벨 파일명을 바꾸세요.

In [ ]:
labels = ANNOTATIONS / "plate_boxes"
label = labels / "a03_sn_f_01_f2.txt"
video_id, frame_index = label_frame(label)
frame = read_rgb(VIDEOS / f"{video_id}.mp4", frame_index)
conf = 0.25
imgsz = 640
pretrained = YOLO(str(WEIGHTS.resolve(strict=True)))
before = predict_rgb(pretrained, frame, conf=conf, imgsz=imgsz)
plot_comparison(frame, label, before, "YOLO26n (COCO)")

## 2. 학습 실행 선택

학습을 시작하려면 `run_training`을 `True`로 바꾸세요. 기존 결과를 열 때에는 `run_dir`을 해당 실험 폴더로 지정합니다.

In [ ]:
run_training = False
run_dir = RUNS / "plate_baseline"
epochs = 20
batch = 4
lr0 = 0.001

if run_training:
    data_yaml = prepare_dataset(VIDEOS, labels, SPLIT, DATASET)
    run_dir = train_baseline(data_yaml, epochs=epochs, imgsz=imgsz, batch=batch,
                             lr0=lr0, device=DEVICE, name=run_dir.name)
print("실험 폴더", run_dir)

## 3. 학습 기록

왼쪽은 학습셋과 검증셋의 손실 합계입니다. 오른쪽은 검증셋의 mAP, 정밀도, 재현율입니다.

In [ ]:
if (run_dir / "results.csv").is_file():
    plot_training(run_dir)
else:
    print("학습 기록이 없습니다. 위 셀에서 학습하거나 기존 실험 폴더를 지정하세요.")

## 4. 같은 장면의 학습 후 예측

검증 성능으로 선택한 `best.pt`를 불러옵니다. 첫 비교와 동일한 프레임, 입력 크기, 신뢰도 기준을 사용합니다.

In [ ]:
best = run_dir / "weights" / "best.pt"
if best.is_file():
    detector = YOLO(str(best))
    after = predict_rgb(detector, frame, conf=conf, imgsz=imgsz)
    plot_comparison(frame, label, after, "YOLO26n (plate)")
else:
    print("이 실험 폴더에 best.pt가 없습니다.")